# Module 9 · 圖像 5：案例 — Dogs vs Cats（現代 PyTorch 管線）

> **本節定位｜整合案例**
> 把圖像 `02–04`（張量前處理、ViT/CLIP、資料組織）整合到經典的
> **Dogs vs Cats** 二分類。示範兩條 2026 路線：
> **(A) CLIP zero-shot（不訓練）**、**(B) 凍結 ViT 特徵 + 邏輯回歸**。
> 全程 CPU 可跑（少量樣本 + 輕量推論），並含離線後備。

## 學習目標
- 用 CLIP 做 **zero-shot** 影像分類（完全不訓練）。
- 用預訓練 ViT 抽特徵 + 傳統分類器（凍結特徵法）。
- 理解「想再更好 → 端到端微調」是下一步（M11）。

In [ ]:
import numpy as np
from PIL import Image

# 準備少量示範影像：優先抓 HF 上的小型 cats_vs_dogs 子集；否則用合成色塊代圖
def load_samples(n_per_class=4):
    try:
        from datasets import load_dataset
        ds = load_dataset("microsoft/cats_vs_dogs", split=f"train[:{n_per_class*40}]")
        cats = [ex["image"] for ex in ds if ex["labels"] == 0][:n_per_class]
        dogs = [ex["image"] for ex in ds if ex["labels"] == 1][:n_per_class]
        return cats, dogs, "HF: microsoft/cats_vs_dogs"
    except Exception:
        rs = np.random.RandomState(0)
        cats = [Image.fromarray((rs.rand(160, 160, 3)*120).astype(np.uint8)) for _ in range(n_per_class)]
        dogs = [Image.fromarray((rs.rand(160, 160, 3)*120+135).clip(0,255).astype(np.uint8)) for _ in range(n_per_class)]
        return cats, dogs, "離線後備合成圖（僅示意流程，不代表真實準確率）"

cats, dogs, source = load_samples()
images = cats + dogs
labels = np.array([0]*len(cats) + [1]*len(dogs))   # 0=cat, 1=dog
print(f"資料來源：{source}")
print(f"樣本數 = {len(images)}（cat={len(cats)}, dog={len(dogs)}）")

## 路線 A：CLIP Zero-shot（不需任何訓練）

In [ ]:
try:
    import torch
    from transformers import CLIPModel, CLIPProcessor
    clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    cproc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    prompts = ["a photo of a cat", "a photo of a dog"]

    preds = []
    for img in images:
        inp = cproc(text=prompts, images=img, return_tensors="pt", padding=True)
        with torch.no_grad():
            logits = clip(**inp).logits_per_image
        preds.append(int(logits.argmax()))
    acc = (np.array(preds) == labels).mean()
    print(f"[CLIP zero-shot] Accuracy = {acc:.3f}（完全沒有訓練！）")
except Exception as e:
    print("（未能下載 CLIP，略過路線 A）", e)

## 路線 B：凍結 ViT 特徵 + 邏輯回歸

In [ ]:
try:
    import torch, timm
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    model.eval()
    cfg = timm.data.resolve_data_config({}, model=model)
    tf = timm.data.create_transform(**cfg)

    feats = []
    with torch.no_grad():
        for img in images:
            x = tf(img.convert("RGB")).unsqueeze(0)
            feats.append(model(x).squeeze(0).numpy())
    X = np.stack(feats)                       # (N, D)
    print(f"ViT 特徵矩陣: {X.shape}  (N, D)")

    clf = LogisticRegression(max_iter=1000)
    if len(images) >= 6:
        scores = cross_val_score(clf, X, labels, cv=2)
        print(f"[凍結 ViT + LogReg] 交叉驗證 Accuracy = {scores.mean():.3f}")
    else:
        clf.fit(X, labels)
        print(f"[凍結 ViT + LogReg] 訓練集 Accuracy = {clf.score(X, labels):.3f}")
except Exception as e:
    print("（未能下載 ViT，略過路線 B）", e)

## 小結與延伸

| 路線 | 是否訓練 | 特點 |
|:--|:--|:--|
| A. CLIP zero-shot | 否 | 最快上手；改類別只要改文字 prompt |
| B. 凍結 ViT + 分類器 | 只訓練分類器 | 用少量標註即可客製 |

**想要最高準確率？** 兩條都是「凍結特徵」。把 ViT **端到端微調**（連骨幹一起學）
通常更強——這是 **Module 11 · `03_image_downstream`** 的內容。